# 01 · Building the data

Stubble-burning fires vs. the crop-residue subsidy in **Punjab & Haryana**. This project
is built from **raw satellite grids and primary government records**, not a pre-cleaned
CSV. Three processed panels drive everything downstream:

| Panel | Grain | Source |
|---|---|---|
| `fire_sage_district_{year,week}` | district × year / week | SAGE-IGP 0.25° daily burned mass (CC0) |
| `weather_district_week` | district × week | Open-Meteo ERA5 archive (keyless) |
| `treatment_district` | district | PPCB MIS + Haryana CRM plan |

**The 2018 wall:** the keyless fire record (SAGE-IGP) ends in 2018 — the CRM scale-up
year — which shapes the whole causal strategy (see notebook 04).

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(ROOT)
if os.path.join(ROOT, 'src') not in sys.path: sys.path.insert(0, os.path.join(ROOT, 'src'))
import pandas as pd, numpy as np
from IPython.display import Image, display
pd.set_option('display.width', 140); pd.set_option('display.max_columns', 40)
print('project root:', ROOT)

project root: E:\New folder (3)


## Fire — SAGE-IGP burned dry matter
Area-weighted from the 0.25° grid to districts (equal-area overlay), so small districts aren't dropped.

In [2]:
fire = pd.read_csv('data/processed/fire_sage_district_year.csv')
print(f'{len(fire)} district-years | {fire.district.nunique()} districts | {fire.year.min()}-{fire.year.max()}')
display(fire.head())
piv = (fire.groupby(['year','state'])['dm_tonnes'].sum()/1e6).unstack('state').round(2)
print('\nState-year dry matter burned (million tonnes):'); display(piv)

301 district-years | 43 districts | 2012-2018


,state,district,year,dm_kg,dm_tonnes
0,Haryana,Ambala,2012,2.774058e+07,27740.575920
1,Haryana,Ambala,2013,4.723451e+07,47234.509395
2,Haryana,Ambala,2014,3.922764e+07,39227.642954
3,Haryana,Ambala,2015,3.840518e+07,38405.176371
4,Haryana,Ambala,2016,3.166164e+07,31661.635961



State-year dry matter burned (million tonnes):


state,Haryana,Punjab
year,,
2012,1.19,6.43
2013,1.62,10.09
2014,1.32,9.29
2015,1.48,11.45
2016,1.86,9.41
2017,1.57,7.80
2018,1.25,7.14


Weekly grain (the prediction target). Burning peaks sharply in **ISO weeks 44–45** (early November).

In [3]:
fw = pd.read_csv('data/processed/fire_sage_district_week.csv')
print(f'{len(fw)} district-weeks'); 
peak = fw.groupby('iso_week')['dm_tonnes'].sum().sort_values(ascending=False).head(5)
print('Top ISO weeks by total burning:'); display((peak/1e3).round(1).rename('DM (kt)'))

2838 district-weeks
Top ISO weeks by total burning:


iso_week
45    26342.1
44    19954.1
46    10310.9
43     7957.6
42     3416.9
Name: DM (kt), dtype: float64

## Weather — Open-Meteo ERA5 (keyless)
Fetched for all 43 districts in one multi-location call per year (~7 requests total).

In [4]:
wx = pd.read_csv('data/processed/weather_district_week.csv')
print(f'{len(wx)} district-weeks | {wx.year.min()}-{wx.year.max()}')
display(wx.head())

4171 district-weeks | 2012-2018


,state,district,year,iso_week,tmax_mean,tmin_mean,precip_sum,rain_days,dry_days,wind_max_mean,wind_calm_days,et0_sum,n_days
0,Haryana,Ambala,2012,35,31.800000,24.500000,14.8,2,0,10.400000,1,7.02,2
1,Haryana,Ambala,2012,36,30.500000,24.628571,45.9,6,0,14.471429,1,24.80,7
2,Haryana,Ambala,2012,37,30.671429,24.585714,25.3,5,0,10.685714,3,21.40,7
3,Haryana,Ambala,2012,38,29.828571,21.714286,29.3,3,4,14.442857,1,26.92,7
4,Haryana,Ambala,2012,39,31.042857,19.971429,0.0,0,7,14.785714,0,31.82,7


## Treatment — harmonised CRM dose

Punjab reports **actual machines delivered** (cumulative 2018-22); Haryana reports
**2018-19 targets**. Not level-comparable, so intensity is harmonised to a **within-state
z-score** (`dose_z`) and share. District names are crosswalked to the geography layer
(1:1, validated).

In [5]:
tr = pd.read_csv('data/processed/treatment_district.csv')
for st in ['Punjab','Haryana']:
    s = tr[tr.state==st]
    print(f"{st}: {int(s.crm_machines.sum()):,} machines ({s.source_kind.iloc[0]})")
display(tr.sort_values('crm_machines', ascending=False)[['state','district','crm_machines','crm_share','dose_z']].head(8))

Punjab: 90,422 machines (actual_machines_cum_2018_22)
Haryana: 5,563 machines (target_machines_2018_19)


,state,district,crm_machines,crm_share,dose_z
21,Punjab,Muktsar,8002.0,0.088496,1.791855
22,Punjab,Sangrur,7538.0,0.083365,1.578227
23,Punjab,Bathinda,6586.0,0.072836,1.139921
24,Punjab,Firozpur,6314.0,0.069828,1.014691
25,Punjab,Mansa,5527.0,0.061125,0.652352
26,Punjab,Ludhiana,5497.0,0.060793,0.638540
27,Punjab,Jalandhar,5230.0,0.057840,0.515612
28,Punjab,Fazilka,4968.0,0.054942,0.394986


**Takeaway:** three clean, keyless panels on a common 43-district grid — ready for EDA, prediction, and the causal design.